In [45]:
import os
import json
from typing import List, Dict, Any

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv

load_dotenv()

mcp_server_path = "E:\Python Projects\langchain-and-langgraph\mcp-crypto-server\mcp_server.py"
print("Setup complete!")

Setup complete!


<>:13: SyntaxWarning: invalid escape sequence '\P'
<>:13: SyntaxWarning: invalid escape sequence '\P'
C:\Users\sk335\AppData\Local\Temp\ipykernel_1216\2630838513.py:13: SyntaxWarning: invalid escape sequence '\P'
  mcp_server_path = "E:\Python Projects\langchain-and-langgraph\mcp-crypto-server\mcp_server.py"


# Tool Discovery: Building Our MCP Host
The first step in building our custom MCP implementation is to create a host that can discover what tools are available from our MCP server. Our host will act as the intermediary between the user, the AI, and the available tools - similar to how Claude Desktop functions, but under our complete control.

Let's implement a function to connect to our MCP server and discover its tools:

In [46]:
import subprocess
import sys
import os

async def discover_tools():
    BLUE = "\033[94m"
    GREEN = "\033[92m"
    RESET = "\033[0m"
    SEP = "=" * 40

    server_params = StdioServerParameters(
        command="python",
        args=[mcp_server_path]
    )

    print(f"{BLUE}{SEP}\n DISCOVERY PHASE: Connecting to MCP SERVER....{RESET}")

    # ✅ Pass errlog=subprocess.DEVNULL to avoid Jupyter's fake stderr
    async with stdio_client(server_params, errlog=subprocess.DEVNULL) as (read, write):
        async with ClientSession(read, write) as session:
            print(f"{BLUE} Initializing MCP Connection....{RESET}")
            await session.initialize()

            print(f"{BLUE} Discovering available tools....{RESET}")
            tools_result = await session.list_tools()

            tool_info = []
            for tool in tools_result.tools:
                tool_info.append({
                    "name": tool.name,
                    "description": tool.description,
                    "schema": tool.inputSchema
                })

            print(f"{GREEN} Successfully discovered {len(tool_info)} tools{RESET}")
            print(f"{SEP}")
            return tool_info


print("Tool discovery function defined")


Tool discovery function defined


In [47]:
# Test the tool discovery function
tools = await discover_tools()
print(f"Discovered {len(tools)} tools:")
for i, tool in enumerate(tools, 1):
    print(f"{i}. {tool['name']}: {tool['description']}")

 DISCOVERY PHASE: Connecting to MCP SERVER....
 Initializing MCP Connection....
 Discovering available tools....
 Successfully discovered 1 tools
Discovered 1 tools:
1. get_crypto_price: 
    Get the current price of a cryptocurrency in a specified currency.
    
    Parameters:
    - crypto_id: The ID of the cryptocurrency (e.g., 'bitcoin', 'ethereum')
    - currency: The currency to display the price in (default: 'usd')
    
    Returns:
    - Current price information as a formatted string
    


# Tool Execution: Implementing Our MCP Client
Now that our host can discover available tools, we need to implement the client component that can execute them. Unlike third-party tools that might have this functionality built-in, we're creating our own client to execute MCP tools with complete control and transparency:

In [48]:
async def execute_tool(tool_name: str, arguments: Dict[str, Any]):
    BLUE = "\033[94m"
    GREEN = "\033[92m"
    YELLOW = "\033[93m"
    RESET = "\033[0m"
    SEP = "-" * 40

    # ✅ Fix 1: Use the mcp-crypto-server's venv Python
    server_python = r"e:\Python Projects\langchain-and-langgraph\mcp-crypto-server\.venv\Scripts\python.exe"

    server_params = StdioServerParameters(
        command=server_python,
        args=[mcp_server_path],
    )

    print(f"{YELLOW}{SEP}")
    print(f"⚙️ EXECUTION PHASE: Running tool '{tool_name}'")
    print(f"📋 Arguments: {json.dumps(arguments, indent=2)}")
    print(f"{SEP}{RESET}")

    # ✅ Fix 2: errlog to avoid Jupyter's fake stderr
    with open("mcp_server_errors.log", "wb") as errlog:
        async with stdio_client(server_params, errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                
                print(f"{BLUE}📡 Sending request to MCP server...{RESET}")
                result = await session.call_tool(tool_name, arguments)
                
                print(f"{GREEN}✅ Tool execution complete{RESET}")
                
                result_preview = str(result)
                if len(result_preview) > 150:
                    result_preview = result_preview[:147] + "..."
                    
                print(f"{BLUE}📊 Result: {result_preview}{RESET}")
                print(f"{SEP}")
                
                return result

print("Tool execution function defined")


Tool execution function defined


# Integrating AI with Our MCP Implementation
With our host and client components in place, we now need to integrate them with an AI system that can make intelligent decisions about tool usage. This is the "brains" of our custom MCP host, and it needs to:

- Understand when a tool is needed based on user input
- Choose the appropriate tool for the task
- Format the arguments correctly
- Process and explain the results

Let's implement a function that orchestrates this entire process:

In [49]:
async def query_claude(prompt: str, tool_info: List[Dict], previous_messages=None):
    """
    Send a query to Claude and process the response.
    
    Args:
        prompt: User's query
        tool_info: Information about available tools
        previous_messages: Previous messages for maintaining context
        
    Returns:
        Claude's response, potentially after executing tools
    """

    # ANSI color codes for better log visibility
    BLUE = "\033[94m"
    GREEN = "\033[92m"
    YELLOW = "\033[93m"
    PURPLE = "\033[95m"
    RESET = "\033[0m"
    SEP = "=" * 40

    if previous_messages is None:
        previous_messages = []

    print(f"{PURPLE}{SEP}")
    print("🧠 REASONING PHASE: Processing query with Claude")
    print(f"Query: \"{prompt}\"")
    print(f"{SEP}{RESET}")

    # Format tool information for claude
    tool_description = "\n\n".join([
        f"Tool: {tool['name']}\nDescription: {tool['description']}\nSchema: {json.dumps(tool['schema'], indent=2)}"
        for tool in tool_info
    ])

    # Build the system prompt
    system_prompt = f"""You are an AI assistant with access to specialized tools through MCP (Model Context Protocol).
    
        Available tools:
        {tool_description}

        When you need to use a tool, respond with a JSON object in the following format:
        {{
           "tool": "tool_name",
           "arguments": {{
               "arg1": "value1",
               "arg2": "value2"
            }}
        }}

    Do not include any other text when using a tool, just the JSON object.
    For regular responses, simply respond normally.
    """

    # filter out system messages from previous messages
    filtered_messages = [msg for msg in previous_messages if msg["role"] != "system"]

    # build the messages for the conversation (without system message)
    messages = filtered_messages.copy()

    # add the current user query
    messages.append({"role": "user", "content": prompt})

    print(f"{BLUE} Sending request to claude API...{RESET}")

    llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.7)
    # Build LangChain-style messages
    from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
    lc_messages = [SystemMessage(content=system_prompt)]
    for msg in filtered_messages:
       if msg["role"] == "user":
         lc_messages.append(HumanMessage(content=msg["content"]))
       elif msg["role"] == "assistant":
         lc_messages.append(AIMessage(content=msg["content"]))
    lc_messages.append(HumanMessage(content=prompt))
    print(f"{BLUE} Sending request to Gemini API...{RESET}")
    response = await llm.ainvoke(lc_messages)    # ✅ Actually call the model
    gemini_response = response.content           # ✅ .content is a string in LangChain
    print(f"{GREEN} Received response from Gemini{RESET}")

    # Try to extract and parse JSON from the response
    try:
        # Look for JSON pattern in the response
        import re
        json_match = re.search(r'(\{[\s\S]*\})', gemini_response)
        
        if json_match:
            json_str = json_match.group(1)
            print(f"{YELLOW}🔍 Tool usage detected in response{RESET}")
            print(f"{BLUE}📦 Extracted JSON: {json_str}{RESET}")
            
            tool_request = json.loads(json_str)
            
            if "tool" in tool_request and "arguments" in tool_request:
                tool_name = tool_request["tool"]
                arguments = tool_request["arguments"]
                
                print(f"{YELLOW}🔧 Claude wants to use tool: {tool_name}{RESET}")
                
                # Execute the tool using our MCP client
                tool_result = await execute_tool(tool_name, arguments)
                
                # Convert tool result to string if needed
                if not isinstance(tool_result, str):
                    tool_result = str(tool_result)
                
                # Update messages with the tool request and result
                messages.append({"role": "assistant", "content": gemini_response})
                messages.append({"role": "user", "content": f"Tool result: {tool_result}"})
                
                print(f"{PURPLE}🔄 Getting Claude's interpretation of the tool result...{RESET}")
                
                # Get Claude's interpretation of the tool result
                llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.7, system=system_prompt, messages=messages)
                
                # ✅ Second call — build updated messages and invoke
                lc_messages.append(AIMessage(content=gemini_response))
                lc_messages.append(HumanMessage(content=f"Tool result: {tool_result}"))

                final_response = await llm.ainvoke(lc_messages)

                return final_response.content, messages

        
    except (json.JSONDecodeError, KeyError, AttributeError) as e:
        print(f"{YELLOW}⚠️ No tool usage detected in response: {str(e)}{RESET}")
    
    print(f"{GREEN}✅ Response ready{RESET}")
    print(f"{SEP}")
    
    return gemini_response, messages

print("Claude query function defined")



    

Claude query function defined


In [50]:
# Run a single query using the tools from your MCP server
query = "What is the current price of Bitcoin?"
print(f"Sending query: {query}")

response, messages = await query_claude(query, tools)
print(f"\nAssistant's response:\n{response}")

Sending query: What is the current price of Bitcoin?
🧠 REASONING PHASE: Processing query with Claude
Query: "What is the current price of Bitcoin?"
 Sending request to claude API...
 Sending request to Gemini API...


ChatGoogleGenerativeAIError: Error calling model 'gemini-2.0-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 24.599388206s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url